# MLflow smoke test

Тестирование подключения к MLFLow из Jupyter Notebook

## Configuration

Edit the next cell, then run all cells top to bottom.

In [2]:
# Хак, чтобы добавить корень проекта в path для импорта модулей
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[1]))
sys.path

['/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python312.zip',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12/lib-dynload',
 '',
 '/home/fiberfox/Projects/HSEAIMag2025/stocks-advisor/.venv/lib/python3.12/site-packages',
 '/home/fiberfox/Projects/HSEAIMag2025',
 '/home/fiberfox/Projects/HSEAIMag2025']

In [4]:
# Код для загрузки конфига и подключения к MLFlow.

import os
import sys
from datetime import UTC, datetime
from pathlib import Path
from typing import Literal

import mlflow
from dotenv import load_dotenv


def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root (pyproject.toml). Open this notebook from the repo.")

def setup(environment: Literal["local", "prod"], experiment: str | None = None):
    CONFIG_BY_ENV = {
        "local": "config_local.toml",
        "prod": "config.toml",
    }

    if environment not in CONFIG_BY_ENV:
        raise ValueError(f"ENV must be one of {list(CONFIG_BY_ENV)}; got {environment!r}")

    REPO_ROOT = find_repo_root()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    os.chdir(REPO_ROOT)

    load_dotenv(REPO_ROOT / ".env")

    os.environ["APP_CONFIG"] = CONFIG_BY_ENV[environment]

    from app.config.settings import get_settings

    get_settings.cache_clear()

    settings = get_settings().mlflow
    
    from app.mlflow import configure_mlflow
    configure_mlflow(experiment)

    resolved_experiment = experiment or settings.default_experiment
    print(f"Environment: {environment}")
    print(f"APP_CONFIG: {os.environ['APP_CONFIG']}")
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    print(f"S3 endpoint: {settings.s3_endpoint_url}")
    print(f"Experiment: {resolved_experiment}")

In [5]:
setup(environment='prod', experiment='test')

Environment: prod
APP_CONFIG: config.toml
Tracking URI: http://localhost:5050
S3 endpoint: http://localhost:9050
Experiment: test


In [6]:
run_name = f"smoke-test-{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_param("source", "mlflow_smoke_test")
    mlflow.log_metric("ping", 1.0)
    mlflow.log_dict({"status": "ok"}, "smoke.json")

print(f"Run ID: {run.info.run_id}")
print(f"Run name: {run_name}")
print("Smoke test passed.")

🏃 View run smoke-test-20260529-212320 at: http://localhost:5050/#/experiments/2/runs/209c34b5eaae4d7b8c1b1a8b5b45ca3b
🧪 View experiment at: http://localhost:5050/#/experiments/2
Run ID: 209c34b5eaae4d7b8c1b1a8b5b45ca3b
Run name: smoke-test-20260529-212320
Smoke test passed.
